# AIRPATH-AI — Milestone 2B XGBoost forecasting

This notebook presents the frozen validation and locked-test outputs. It intentionally does **not** rerun model selection or test evaluation when executed. The one-shot experiment entry point is `python3 -m src.xgboost_forecasting`.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.xgboost_forecasting import HourlyStationForecaster

TABLES = ROOT / "reports" / "tables"
MODELS = ROOT / "data" / "processed" / "models"

In [ ]:
metadata = json.loads((MODELS / "metadata.json").read_text())
metrics = pd.read_csv(TABLES / "xgboost_metrics.csv")
search = pd.read_csv(TABLES / "xgboost_validation_search.csv")
improvements = pd.read_csv(TABLES / "xgboost_improvement_over_persistence.csv")
imputation = pd.read_csv(TABLES / "xgboost_weather_imputation.csv")
importance = pd.read_csv(TABLES / "xgboost_feature_importance.csv")
shap_importance = pd.read_csv(TABLES / "xgboost_shap_importance.csv")

display(metadata)
display(metrics.loc[metrics["Station_No"].astype(str).eq("ALL")])

In [ ]:
display(search.sort_values(["horizon_hours", "validation_mae"]))
display(improvements.loc[
    improvements["Station_No"].astype(str).eq("ALL")
])
display(imputation)
display(importance.groupby("horizon_hours", group_keys=False).head(10))
display(shap_importance.groupby("horizon_hours", group_keys=False).head(10))

## Target-time API

The serialized `HourlyStationForecaster` exposes `predict_pm25(station_or_location, target_time, *, prediction_time, pm25_lags, temperature=None, humidity=None)`. It supports only known monitored stations and exact 1–3 hour targets. Geographic road locations and sub-hourly times are rejected until separately validated spatial and higher-resolution layers exist.

In [ ]:
forecaster = HourlyStationForecaster.load(
    MODELS / "hourly_station_forecaster.joblib"
)
print(type(forecaster).__name__)
print("Selected version:", forecaster.version)
print("Supported monitored stations:", forecaster.station_ids)
print("Supported horizons:", sorted(forecaster.models))

The locked test results must not be used for further tuning. No XGBoost experiment is rerun from this presentation notebook.